# Enhancing Time Series Momentum Strategies Using Deep Neural Networks

**Authors**: Bryan Lim, Stefan Zohren, Stephen Roberts

**Published**: 2019-01-01

[Link to Paper](https://doi.org/10.2139/ssrn.3369195)

**Abstract**: This notebook follows the CRISP-TIQ 6-phase structure to demonstrate a time series momentum strategy using deep learning enhancements. We will use yfinance for data retrieval and perform analysis using pandas, numpy, matplotlib, and scipy.

In [ ]:
!pip install yfinance

## Phase 1: Configuration

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration parameters
TICKER = 'AAPL'
START_DATE = '2010-01-01'
END_DATE = '2023-01-01'
LOOKBACK_PERIOD = 20


## Phase 2: Data Download and Feature Engineering

In [ ]:
# Download data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Feature engineering
data['Returns'] = data['Adj Close'].pct_change()
data['Rolling Mean'] = data['Adj Close'].rolling(window=LOOKBACK_PERIOD).mean()
data['Rolling Std'] = data['Adj Close'].rolling(window=LOOKBACK_PERIOD).std()
data.dropna(inplace=True)


## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
# Signal generation
data['Signal'] = np.where(data['Adj Close'] > data['Rolling Mean'], 1, -1)

# Portfolio construction
data['Strategy Returns'] = data['Signal'].shift(1) * data['Returns']


## Phase 4: Vectorized Backtest

In [ ]:
# Calculate cumulative returns
data['Cumulative Market Returns'] = (1 + data['Returns']).cumprod()
data['Cumulative Strategy Returns'] = (1 + data['Strategy Returns']).cumprod()


## Phase 5: Performance Metrics

In [ ]:
# Calculate performance metrics
sharpe_ratio = data['Strategy Returns'].mean() / data['Strategy Returns'].std() * np.sqrt(252)
sortino_ratio = data['Strategy Returns'].mean() / data[data['Strategy Returns'] < 0]['Strategy Returns'].std() * np.sqrt(252)
calmar_ratio = data['Strategy Returns'].mean() * 252 / data['Cumulative Strategy Returns'].expanding().max().sub(data['Cumulative Strategy Returns']).max()
max_drawdown = data['Cumulative Strategy Returns'].expanding().max().sub(data['Cumulative Strategy Returns']).max()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Cumulative Market Returns'], label='Market')
plt.plot(data['Cumulative Strategy Returns'], label='Strategy')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

# Print performance metrics
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")


## Phase 6: Monitoring Stub

This phase would typically involve setting up real-time data feeds and alerts for strategy monitoring. This is a placeholder for future development.